# Lily 1.5B GRPO Model Inference — Google Colab
**Interactive Reasoning & Inference for `abhinav0231/Lily-1.5B`**

This notebook runs in Google Colab. It loads the trained GRPO model `abhinav0231/Lily-1.5B` using Unsloth's 2x fast inference engine (`FastLanguageModel.for_inference`), parses step-by-step thinking `<think>...</think>` tags and final `<answer>...</answer>` tags, and includes an interactive query loop for testing.

## Cell 1 — Install Unsloth & Dependencies

In [ ]:
# ==============================================================================
# Cell 1 — Install Unsloth & Hugging Face Libraries
# ==============================================================================
# Install Unsloth fast inference engine and huggingface_hub for model downloading
!pip install unsloth huggingface_hub -q

## Cell 2 — Load Trained GRPO Model with Unsloth

In [ ]:
# ==============================================================================
# Robust Authentication (Hugging Face)
# ==============================================================================
import os
try:
    from huggingface_hub import login, get_token
except ImportError:
    from huggingface_hub import login, HfFolder
    get_token = HfFolder.get_token

# Retrieve HF Token from all possible sources (os.environ, Colab Secrets, or cached token)
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN", "")
if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN") or ""
    except Exception:
        pass

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    cached_token = get_token()
    if cached_token:
        HF_TOKEN = cached_token

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✅ Authenticated with Hugging Face")
    except Exception as e:
        print(f"⚠️ Hugging Face authentication note: {e}")
else:
    print("ℹ️ HF_TOKEN not provided. Proceeding (public datasets/models remain accessible).")

# WandB Authentication
try:
    import wandb
    WANDB_TOKEN = os.environ.get("WANDB_API_KEY", "")
    if WANDB_TOKEN and WANDB_TOKEN != "YOUR_WANDB_KEY_HERE":
        wandb.login(key=WANDB_TOKEN, relogin=True)
        os.environ["WANDB_API_KEY"] = WANDB_TOKEN
        print("✅ Authenticated with Weights & Biases")
    else:
        print("ℹ️ WANDB_API_KEY not found. WandB tracking will operate in offline/disabled mode.")
        os.environ["WANDB_DISABLED"] = "true"
except ImportError:
    print("ℹ️ WandB module not installed. Operating without WandB tracking.")
    os.environ["WANDB_DISABLED"] = "true"


## Cell 3 — Inference Function with Reasoning CoT Parser

In [ ]:
# ==============================================================================
# Cell 3 — Prompt Formatting, Generation Loop & Regular Expression CoT Parser
# ==============================================================================
import re

# Standardized System Prompt to trigger structural step-by-step reasoning
SYSTEM_PROMPT = (
    "You are a precise, helpful assistant. "
    "Always reason step by step inside <think></think> tags, "
    "then write your final answer inside <answer></answer> tags."
)

def ask(question, max_new_tokens=1024, temperature=0.7):
    """
    Formats user prompt into ChatML structure, tokenizes, generates response via PyTorch,
    and returns newly generated completion string.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    
    # Format and tokenize input query with generation prompt prompt delimiter
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt",
    ).to("cuda")

    # Generate model response using sampling parameters
    output_ids = model.generate(
        input_ids,
        max_new_tokens = max_new_tokens,
        temperature    = temperature,
        top_p          = 0.95,
        do_sample      = True if temperature > 0 else False,
        pad_token_id   = tokenizer.eos_token_id,
    )

    # Decode only the newly generated tokens (excluding prompt tokens)
    response = tokenizer.decode(
        output_ids[0][input_ids.shape[-1]:],
        skip_special_tokens = True
    )
    return response

def parse_and_print(response):
    """
    Extracts and prints 'think_body' (<think>...</think>) and 'answer_body' (<answer>...</answer>)
    using regular expressions.
    """
    think_m  = re.search(r"<think>(.*?)</think>", response, re.DOTALL)
    answer_m = re.search(r"<answer>(.*?)</answer>", response, re.DOTALL)
    
    print("🧠 REASONING (<think>):")
    if think_m:
        print(think_m.group(1).strip())
    else:
        print("No <think> tag found. Full output:")
        print(response)
        
    print("\n🎯 FINAL ANSWER (<answer>):")
    if answer_m:
        print(answer_m.group(1).strip())
    else:
        print(response.split("</think>")[-1].strip())

print("✅ Inference & Parsing functions ready")

## Cell 4 — Run Sample Test Queries (Math, Logic, Coding)

In [ ]:
# ==============================================================================
# Cell 4 — Test Execution Across Mathematics, Physics & Coding Tasks
# ==============================================================================
test_questions = [
    "What is 15% of 840?",
    "If a train travels 120 km in 1.5 hours, what is its speed in m/s?",
    "A bat and a ball cost $1.10 together. The bat costs $1.00 more than the ball. How much does the ball cost?",
    "Write a Python function to check if a string is a palindrome."
]

# Execute generation and parse output tags for each test query
for q in test_questions:
    print(f"\n{'='*70}")
    print(f"❓ QUESTION: {q}")
    print(f"{'='*70}")
    raw_out = ask(q, temperature=0.7)
    parse_and_print(raw_out)

## Cell 5 — Interactive Query Testing Loop

In [ ]:
# ==============================================================================
# Cell 5 — Interactive User Query Loop
# ==============================================================================
print("Type your question below (or type 'exit' to stop):\n")
while True:
    user_query = input("Enter Query: ")
    if user_query.strip().lower() in ["exit", "quit", "q"]:
        print("Exiting interactive loop.")
        break
    if not user_query.strip():
        continue
        
    print(f"\n{'='*70}")
    print(f"❓ QUESTION: {user_query}")
    print(f"{'='*70}")
    raw_out = ask(user_query, temperature=0.7)
    parse_and_print(raw_out)
    print("\n")